In [1]:
import pandas as pd
import numpy as np

df_b = pd.read_csv('../data/processed/bengaluru_cleaned.csv')
df_g = pd.read_csv('../data/processed/global_cleaned.csv')

print(f"Bengaluru cleaned : {df_b.shape}")
print(f"Global cleaned    : {df_g.shape}")
print(f"\nBengaluru columns : {df_b.columns.tolist()}")
print(f"Global columns    : {df_g.columns.tolist()}")

Bengaluru cleaned : (41410, 11)
Global cleaned    : (6513, 14)

Bengaluru columns : ['name', 'online_order', 'book_table', 'rate', 'votes', 'location', 'rest_type', 'cuisines', 'cost_for_two', 'listed_type', 'listed_city']
Global columns    : ['name', 'city', 'address', 'location', 'longitude', 'latitude', 'cuisines', 'cost_for_two', 'book_table', 'online_order', 'price_range', 'rate', 'rating_text', 'votes']


In [2]:
print("=" * 50)
print("BENGALURU — STATISTICAL SUMMARY")
print("=" * 50)
print(df_b[['rate', 'cost_for_two', 'votes']].describe().round(2))

print("\n" + "=" * 50)
print("GLOBAL — STATISTICAL SUMMARY")
print("=" * 50)
print(df_g[['rate', 'cost_for_two', 'votes']].describe().round(2))

BENGALURU — STATISTICAL SUMMARY
           rate  cost_for_two     votes
count  41410.00      41410.00  41410.00
mean       3.70        603.31    351.79
std        0.44        464.36    882.77
min        1.80         40.00      0.00
25%        3.40        300.00     21.00
50%        3.70        500.00     73.00
75%        4.00        700.00    277.00
max        4.90       6000.00  16832.00

GLOBAL — STATISTICAL SUMMARY
          rate  cost_for_two     votes
count  6513.00       6513.00   6513.00
mean      3.35        715.91    181.99
std       0.50        647.00    485.06
min       1.80          0.00      4.00
25%       3.00        350.00     16.00
50%       3.30        500.00     49.00
75%       3.70        800.00    148.00
max       4.90       8000.00  10934.00


In [3]:
print("=" * 50)
print("Q1: RATING DISTRIBUTION — BENGALURU")
print("=" * 50)

# Overall stats
print(f"Mean Rating   : {df_b['rate'].mean():.2f}")
print(f"Median Rating : {df_b['rate'].median():.2f}")
print(f"Std Deviation : {df_b['rate'].std():.2f}")

# Rating buckets
bins = [0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
labels = ['Poor(<2.5)', 'Below Avg(2.5-3.0)', 
          'Average(3.0-3.5)', 'Good(3.5-4.0)',
          'Very Good(4.0-4.5)', 'Excellent(4.5+)']

df_b['rating_category'] = pd.cut(df_b['rate'], 
                                   bins=bins, 
                                   labels=labels)

print("\nRating Category Distribution:")
print(df_b['rating_category'].value_counts().sort_index())
print("\nRating Category % :")
print((df_b['rating_category'].value_counts(normalize=True)
       .sort_index() * 100).round(2))

Q1: RATING DISTRIBUTION — BENGALURU
Mean Rating   : 3.70
Median Rating : 3.70
Std Deviation : 0.44

Rating Category Distribution:
rating_category
Poor(<2.5)              287
Below Avg(2.5-3.0)     2973
Average(3.0-3.5)      10935
Good(3.5-4.0)         18059
Very Good(4.0-4.5)     8568
Excellent(4.5+)         588
Name: count, dtype: int64

Rating Category % :
rating_category
Poor(<2.5)             0.69
Below Avg(2.5-3.0)     7.18
Average(3.0-3.5)      26.41
Good(3.5-4.0)         43.61
Very Good(4.0-4.5)    20.69
Excellent(4.5+)        1.42
Name: proportion, dtype: float64


In [4]:
print("=" * 50)
print("Q2: ONLINE ORDER vs RATING")
print("=" * 50)

online_rating = df_b.groupby('online_order')['rate'].agg(
    ['mean', 'median', 'count']
).round(2)

online_rating.index = ['No Online Order', 'Has Online Order']
print(online_rating)

# Business insight check
diff = (df_b[df_b['online_order']==1]['rate'].mean() - 
        df_b[df_b['online_order']==0]['rate'].mean())

print(f"\nRating difference: {diff:.2f}")
print(f"Business Insight: Restaurants WITH online ordering rate "
      f"{'higher' if diff > 0 else 'lower'} by {abs(diff):.2f} points")

Q2: ONLINE ORDER vs RATING
                  mean  median  count
No Online Order   3.66     3.7  14212
Has Online Order  3.72     3.8  27198

Rating difference: 0.06
Business Insight: Restaurants WITH online ordering rate higher by 0.06 points


In [5]:
print("=" * 50)
print("Q3: COST vs RATING CORRELATION")
print("=" * 50)

correlation = df_b['cost_for_two'].corr(df_b['rate'])
print(f"Pearson Correlation (Cost vs Rating): {correlation:.4f}")

# Interpret the correlation
if correlation > 0.5:
    insight = "Strong positive — higher cost = higher rating"
elif correlation > 0.3:
    insight = "Moderate positive — some relationship exists"
elif correlation > 0:
    insight = "Weak positive — minimal relationship"
else:
    insight = "Negative/No relationship — cost doesn't drive rating"

print(f"Interpretation: {insight}")

# Cost bracket analysis
df_b['cost_bracket'] = pd.cut(df_b['cost_for_two'],
                               bins=[0, 300, 600, 900, 1200, 10000],
                               labels=['Budget(<300)', 'Economy(300-600)',
                                      'Mid(600-900)', 'Premium(900-1200)',
                                      'Luxury(1200+)'])

print("\nAvg Rating by Cost Bracket:")
print(df_b.groupby('cost_bracket')['rate'].mean().round(2))

Q3: COST vs RATING CORRELATION
Pearson Correlation (Cost vs Rating): 0.3852
Interpretation: Moderate positive — some relationship exists

Avg Rating by Cost Bracket:
cost_bracket
Budget(<300)         3.57
Economy(300-600)     3.62
Mid(600-900)         3.79
Premium(900-1200)    3.94
Luxury(1200+)        4.16
Name: rate, dtype: float64


C:\Users\Yash Chavan\AppData\Local\Temp\ipykernel_24052\1542583673.py:28: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df_b.groupby('cost_bracket')['rate'].mean().round(2))


In [6]:
print("=" * 50)
print("Q4: LOCATION vs RATING")
print("=" * 50)

location_analysis = df_b.groupby('location').agg(
    avg_rating    = ('rate', 'mean'),
    total_restaurants = ('name', 'count'),
    avg_cost      = ('cost_for_two', 'mean')
).round(2)

# Only locations with 50+ restaurants (statistically significant)
location_analysis = location_analysis[
    location_analysis['total_restaurants'] >= 50
]

print("TOP 10 LOCATIONS by Rating:")
print(location_analysis.sort_values('avg_rating', 
      ascending=False).head(10))

print("\nBOTTOM 10 LOCATIONS by Rating:")
print(location_analysis.sort_values('avg_rating', 
      ascending=True).head(10))

Q4: LOCATION vs RATING
TOP 10 LOCATIONS by Rating:
                       avg_rating  total_restaurants  avg_cost
location                                                      
Lavelle Road                 4.14                481   1365.38
Koramangala 3rd Block        4.02                191    834.82
St. Marks Road               4.02                343    883.67
Koramangala 5th Block        4.01               2297    680.71
Church Street                3.99                546    839.84
Koramangala 4th Block        3.92                841    758.32
Cunningham Road              3.90                475    867.16
Residency Road               3.86                604   1030.05
MG Road                      3.85                793   1244.51
Koramangala 7th Block        3.85               1055    604.36

BOTTOM 10 LOCATIONS by Rating:
                    avg_rating  total_restaurants  avg_cost
location                                                   
Bommanahalli              3.19           

In [7]:
print("=" * 50)
print("Q5: TOP CUISINES IN BENGALURU")
print("=" * 50)

# Cuisines column has multiple values per row
# Example: "North Indian, Chinese, Mughlai"
# We need to split and count each individually

cuisine_series = df_b['cuisines'].str.split(',').explode().str.strip()

print("Top 15 Cuisines by Restaurant Count:")
print(cuisine_series.value_counts().head(15))

print("\nTop 10 Cuisines by Avg Rating:")
df_b['primary_cuisine'] = df_b['cuisines'].str.split(',').str[0].str.strip()
print(df_b.groupby('primary_cuisine')['rate']
      .agg(['mean','count'])
      .query('count >= 50')
      .sort_values('mean', ascending=False)
      .head(10)
      .round(2))

Q5: TOP CUISINES IN BENGALURU
Top 15 Cuisines by Restaurant Count:
cuisines
North Indian    17310
Chinese         12930
South Indian     6369
Fast Food        6340
Continental      5202
Biryani          5060
Cafe             4785
Desserts         4512
Beverages        3839
Italian          3188
Street Food      2219
Bakery           2023
Pizza            1877
Burger           1845
Seafood          1649
Name: count, dtype: int64

Top 10 Cuisines by Avg Rating:
                 mean  count
primary_cuisine             
Modern Indian    4.31    109
European         4.26    229
Mediterranean    4.20    101
Japanese         4.19    112
American         4.16    441
Asian            4.15    370
Goan             4.04     77
Thai             4.04    128
Lebanese         4.02     65
BBQ              4.00    108


In [8]:
print("=" * 50)
print("Q6: TABLE BOOKING vs RATING")
print("=" * 50)

table_rating = df_b.groupby('book_table')['rate'].agg(
    ['mean', 'median', 'count']
).round(2)

table_rating.index = ['No Table Booking', 'Has Table Booking']
print(table_rating)

# Average cost comparison too
print("\nAvg Cost by Table Booking:")
print(df_b.groupby('book_table')['cost_for_two'].mean().round(2))

diff_table = (df_b[df_b['book_table']==1]['rate'].mean() - 
              df_b[df_b['book_table']==0]['rate'].mean())

print(f"\nRating difference: {diff_table:.2f}")
print(f"Business Insight: Table booking restaurants rate "
      f"{'higher' if diff_table > 0 else 'lower'} "
      f"by {abs(diff_table):.2f} points")

Q6: TABLE BOOKING vs RATING
                   mean  median  count
No Table Booking   3.62     3.7  35106
Has Table Booking  4.14     4.2   6304

Avg Cost by Table Booking:
book_table
0     482.43
1    1276.49
Name: cost_for_two, dtype: float64

Rating difference: 0.52
Business Insight: Table booking restaurants rate higher by 0.52 points
